# 01. 線形代数の復習 - 情報幾何への橋渡し

情報幾何で特に重要な線形代数の概念を復習します。

## 本ノートブックの目標
- 内積と計量の関係を理解する
- 正定値行列の意味を確認する
- 双対空間の概念に触れる
- これらがFisher情報行列とどう関連するかを把握する

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

# 日本語フォント設定（環境に応じて調整）
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (10, 6)

---
## 1. 内積と計量

### 標準内積
ユークリッド空間 $\mathbb{R}^n$ での標準内積：
$$\langle u, v \rangle = u^\top v = \sum_{i=1}^n u_i v_i$$

### 一般の内積（計量）
正定値対称行列 $G$ を用いた内積：
$$\langle u, v \rangle_G = u^\top G v$$

**🔗 情報幾何との接続**: Fisher情報行列 $I(\theta)$ がこの $G$ の役割を果たします。

In [ ]:
# 内積の比較: 標準 vs 計量付き

# 2つのベクトル
u = np.array([1, 0])
v = np.array([1, 1])

# 標準内積
inner_standard = u @ v
print(f"標準内積 <u, v> = {inner_standard}")

# 計量行列 G を用いた内積
# 例: G = [[2, 0], [0, 1]] → x方向の「重み」が2倍
G = np.array([[2, 0], 
              [0, 1]])
inner_G = u @ G @ v
print(f"計量付き内積 <u, v>_G = {inner_G}")

In [ ]:
def plot_metric_comparison():
    """
    標準内積と計量付き内積での「単位円」の違いを可視化
    ||v||² = 1 となるベクトルの集合
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    theta = np.linspace(0, 2*np.pi, 100)
    
    # 左: 標準内積での単位円
    ax1 = axes[0]
    x_std = np.cos(theta)
    y_std = np.sin(theta)
    ax1.plot(x_std, y_std, 'b-', linewidth=2, label='Standard unit circle')
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax1.set_title('Standard metric: $||v||^2 = v^T v = 1$')
    ax1.set_aspect('equal')
    ax1.grid(True, alpha=0.3)
    ax1.axhline(0, color='k', linewidth=0.5)
    ax1.axvline(0, color='k', linewidth=0.5)
    
    # 右: 計量 G = [[2, 0], [0, 1]] での「単位円」
    # v^T G v = 1 → 2x² + y² = 1 (楕円)
    ax2 = axes[1]
    x_G = np.cos(theta) / np.sqrt(2)  # 2x² = cos²θ → x = cosθ/√2
    y_G = np.sin(theta)
    ax2.plot(x_G, y_G, 'r-', linewidth=2, label='Metric G unit "circle"')
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')
    ax2.set_title('Metric G=diag(2,1): $||v||_G^2 = v^T G v = 1$')
    ax2.set_aspect('equal')
    ax2.grid(True, alpha=0.3)
    ax2.axhline(0, color='k', linewidth=0.5)
    ax2.axvline(0, color='k', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()
    
    print("""    
【ポイント】
- 計量が変わると「長さ1」の意味が変わる
- G = diag(2, 1) では x方向の移動が「重く」なる（同じ座標変化でも長さが大きい）
- Fisher計量では σ が小さい領域での移動が「重く」なる
""")

plot_metric_comparison()

---
## 2. 正定値行列

### 定義
対称行列 $G$ が**正定値 (positive definite)** であるとは：
$$v^\top G v > 0 \quad (\forall v \neq 0)$$

### 同値条件
- すべての固有値が正
- $G = L L^\top$ と分解可能（コレスキー分解）
- 主小行列式がすべて正

**🔗 情報幾何との接続**: Fisher情報行列は常に半正定値、正則な統計モデルでは正定値

In [ ]:
def check_positive_definite(G, name="G"):
    """正定値性のチェック"""
    eigenvalues = np.linalg.eigvalsh(G)
    is_pd = np.all(eigenvalues > 0)
    
    print(f"Matrix {name}:")
    print(G)
    print(f"Eigenvalues: {eigenvalues}")
    print(f"Positive definite: {is_pd}")
    print()
    return is_pd

# 例1: 正定値行列
G1 = np.array([[2, 1], 
               [1, 2]])
check_positive_definite(G1, "G1 (positive definite)")

# 例2: 正規分布のFisher情報行列 (σ=1の場合)
G_fisher = np.array([[1, 0], 
                     [0, 2]])  # I = diag(1/σ², 2/σ²) at σ=1
check_positive_definite(G_fisher, "Fisher info (Gaussian, σ=1)")

# 例3: 正定値でない行列
G3 = np.array([[1, 2], 
               [2, 1]])
check_positive_definite(G3, "G3 (not positive definite)")

---
## 3. 二次形式と楕円

正定値行列 $G$ による二次形式 $v^\top G v = c$ は楕円を定義します。

楕円の形状は $G$ の固有値と固有ベクトルで決まります：
- **固有ベクトル**: 楕円の主軸の方向
- **固有値**: 主軸の長さの逆二乗に比例

In [ ]:
def plot_quadratic_form_ellipse(G, ax, color='blue', label=''):
    """
    二次形式 v^T G v = 1 の楕円を描画
    """
    eigenvalues, eigenvectors = np.linalg.eigh(G)
    
    # 楕円のパラメータ
    angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0]))
    width = 2 / np.sqrt(eigenvalues[0])  # 固有値が大きいほど軸が短い
    height = 2 / np.sqrt(eigenvalues[1])
    
    ellipse = Ellipse((0, 0), width, height, angle=angle,
                      fill=False, edgecolor=color, linewidth=2, label=label)
    ax.add_patch(ellipse)
    
    # 固有ベクトルの方向を表示
    for i in range(2):
        ev = eigenvectors[:, i] / np.sqrt(eigenvalues[i]) * 0.8
        ax.arrow(0, 0, ev[0], ev[1], head_width=0.05, head_length=0.03, 
                 fc=color, ec=color, alpha=0.7)
    
    return eigenvalues, eigenvectors

fig, ax = plt.subplots(figsize=(8, 8))

# 異なる計量での楕円を比較
G1 = np.array([[1, 0], [0, 1]])  # 単位行列 → 円
G2 = np.array([[2, 0], [0, 1]])  # 対角行列 → 軸に沿った楕円
G3 = np.array([[2, 1], [1, 2]])  # 非対角 → 回転した楕円

plot_quadratic_form_ellipse(G1, ax, 'blue', 'I (identity)')
plot_quadratic_form_ellipse(G2, ax, 'red', 'diag(2,1)')
plot_quadratic_form_ellipse(G3, ax, 'green', '[[2,1],[1,2]]')

ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.legend()
ax.set_title('Quadratic forms $v^T G v = 1$ (unit ellipses)')
plt.show()

print("""
【ポイント】
- 計量行列 G が楕円の形を決める
- Fisher情報行列は各点で異なる → 多様体上の各点で楕円の形が変わる
- これが「曲がった空間」の直感的イメージ
""")

---
## 4. 双対空間（発展）

### 定義
ベクトル空間 $V$ の**双対空間** $V^*$ は、$V$ から実数への線形写像の集合：
$$V^* = \{ \phi: V \to \mathbb{R} \mid \phi \text{ is linear} \}$$

### 計量による同一視
計量 $G$ があると、$V$ と $V^*$ を自然に対応づけられる：
$$v \in V \mapsto \phi_v \in V^* \quad \text{where} \quad \phi_v(w) = \langle v, w \rangle_G = v^\top G w$$

座標で書くと：
- ベクトル（反変）: $v^i$ （上付き添字）
- 双対ベクトル（共変）: $v_i = G_{ij} v^j$ （下付き添字）

**🔗 情報幾何との接続**: 
- 接ベクトル（パラメータの変化方向）と余接ベクトル（勾配）
- 自然勾配 = Fisher計量による双対変換

In [ ]:
def demonstrate_dual_space():
    """
    双対空間と計量による対応を具体例で示す
    """
    # ベクトル v
    v = np.array([1, 2])
    
    # 計量行列
    G = np.array([[2, 0], 
                  [0, 1]])
    
    # 双対ベクトル (共変成分)
    v_dual = G @ v
    
    print("Vector v (contravariant):")
    print(f"  v = {v}")
    print(f"\nMetric G:")
    print(G)
    print(f"\nDual vector (covariant):")
    print(f"  v_dual = G @ v = {v_dual}")
    
    # 内積の計算（2つの方法）
    w = np.array([3, 1])
    inner_1 = v @ G @ w
    inner_2 = v_dual @ w
    
    print(f"\nInner product <v, w>_G:")
    print(f"  v^T G w = {inner_1}")
    print(f"  v_dual^T w = {inner_2}")
    print(f"  (Both methods give the same result)")
    
    print("""
【情報幾何への接続】
- 通常の勾配 ∇L は「余接ベクトル」（共変）
- パラメータ更新方向は「接ベクトル」（反変）
- 自然勾配: ∇̃L = I(θ)⁻¹ ∇L （Fisher計量で共変→反変に変換）
""")

demonstrate_dual_space()

---
## 5. 確認問題

### Q1. 計量と内積
計量行列 $G = \begin{pmatrix} 4 & 0 \\ 0 & 1 \end{pmatrix}$ に対して、
ベクトル $v = (1, 2)^\top$ の長さ $||v||_G$ を計算せよ。

### Q2. Fisher情報行列
正規分布 $N(\mu, \sigma^2)$ のFisher情報行列が
$I = \begin{pmatrix} 1/\sigma^2 & 0 \\ 0 & 2/\sigma^2 \end{pmatrix}$
であるとき、$\sigma = 0.5$ での計量楕円の形状を予測せよ。

### Q3. 自然勾配
通常の勾配が $\nabla L = (1, 1)^\top$ のとき、
Fisher情報行列 $I = \begin{pmatrix} 2 & 0 \\ 0 & 1 \end{pmatrix}$ に対する
自然勾配 $\tilde{\nabla} L = I^{-1} \nabla L$ を計算せよ。

In [ ]:
# Q1の解答欄
G = np.array([[4, 0], [0, 1]])
v = np.array([1, 2])

# ||v||_G = sqrt(v^T G v)
norm_v_G = np.sqrt(v @ G @ v)
print(f"Q1: ||v||_G = {norm_v_G}")

In [ ]:
# Q2の解答欄
sigma = 0.5
I_fisher = np.array([[1/sigma**2, 0], 
                     [0, 2/sigma**2]])
print(f"Q2: Fisher info at σ={sigma}:")
print(I_fisher)
print(f"\nI_μμ = {I_fisher[0,0]}, I_σσ = {I_fisher[1,1]}")
print("σが小さいのでFisher情報が大きく、楕円は小さくなる（推定精度が高い）")

In [ ]:
# Q3の解答欄
I = np.array([[2, 0], [0, 1]])
grad_L = np.array([1, 1])

natural_grad = np.linalg.inv(I) @ grad_L
print(f"Q3: Natural gradient = I⁻¹ @ ∇L = {natural_grad}")
print(f"\n通常勾配 (1, 1) に対して、μ方向は半分、σ方向は同じ")
print(f"→ Fisher情報が大きい方向（μ）は更新を抑制")

---
## まとめ

| 線形代数の概念 | 情報幾何での対応 |
|--------------|----------------|
| 内積 $\langle u, v \rangle_G$ | リーマン計量 |
| 正定値行列 | Fisher情報行列の性質 |
| 二次形式の楕円 | 計量楕円（推定精度の可視化） |
| 双対空間 | 接空間と余接空間 |
| 計量による双対変換 | 自然勾配 |

---
**次のノートブック**: `02_probability_statistics.ipynb` - 確率・統計の復習